In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col,
    row_number,
    trim
)
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType


##### Importing from validation_utils and utils package

In [0]:
from validation_utils.test_utils import Validations
from utils.custom_utils import Transformations
tr_obj = Transformations()
va_obj = Validations()

#### Reading bronze table

In [0]:
df = spark.table("lakehouse.bronze.sales_details")

### Silver layer transformations

#### 1. Renaming columns

In [0]:
rename_config = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in rename_config.items():
    df = df.withColumnRenamed(old_name, new_name)

#### 2. Removing records with no order_number
- Since there are no nulls in the order_number so transformation not needed

In [0]:
va_obj.null_check(df, 'order_number')

#### 3. Removing the duplicates of the order_number, product_number and customer_id combination.
- There are no duplicates

In [0]:
va_obj.duplicate_check(df = df, dedup_cols = ['order_number', 'product_number', 'customer_id'], cdc = 'order_date')

#### 4. Trimming
- There are no leading or trailing spaces

In [0]:
va_obj.whitespace_check(df)

#### 5. Date transformation
- Date cannot be integer
- First check if the integer is <= 0, if yes then insert nulls
- Since the standard length of such integers is 8 check for valuse with lesser length, if yes then insert null
- Then finally cast the integer into string and then string into date
- Lastly, check whether order_date > (ship_date or due_date)

In [0]:
df = (
    df.withColumn(
        "order_date",
        F.when((col("order_date") <= 0) | (F.length(col("order_date")) != 8), None)
        .otherwise(F.to_date(col("order_date").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "ship_date",
        F.when((col("ship_date") <= 0) | (F.length(col("ship_date")) != 8), None)
        .otherwise(F.to_date(col("ship_date").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "due_date",
        F.when((col("due_date") <= 0) | (F.length(col("due_date")) != 8), None)
        .otherwise(F.to_date(col("due_date").cast("string"), "yyyyMMdd"))
    )
)

#### 6. Cleaning sales_amount, quantity and price using business logic
- Sales = qualtity * price
- Check whether there are nulls or negatives in the above columns, if yes then apply transformation
- For sales and price, find the value using the above formula
- quantity is neither null nor negative
- In the table, both the sales_amount and price are not null in the same record so using the above formula is good because during the multiplication, if nulls are encountered then those records are simply ignored and left as it is.

In [0]:
df = (
    df.withColumn(
        "sales_amount",
        F.when(
            (col("sales_amount").isNull()) |
                (col("sales_amount") < 0) |
                 (col("sales_amount") != (col("quantity") * F.abs(col("price")))), 
                 col("quantity") * F.abs(col("price"))
        ).otherwise(col("sales_amount"))
    )
    .withColumn(
        "price",
        F.when(
            (col("price").isNull()) | (col("price") < 0),
            F.when(
                col("quantity") != 0,
                (col("sales_amount") / col("quantity")).cast("int")
            ).otherwise(None)
        ).otherwise(col("price"))
    )
)

#### 7. Adding ingestion_ts column

In [0]:
df = tr_obj.add_ingestion_timestamp(df)

#### DataFrame sanity check

In [0]:
df.limit(10).display()

#### 8. Applying upsert
- Since this is a transactional table:
- 1. New records are appended
- 2. It remain immutable i.e the created records are immutable
- 3. It should preserve history
- So, the upsert is not applied in the transactional tables

In [0]:
df.write.format("delta").mode("append").saveAsTable("lakehouse.silver.crm_sales")

#### Silver table sanity check

In [0]:
%sql
select *
from lakehouse.silver.crm_sales